# 04 — Model Evaluation

## Objective
Deep-dive into the AutoML experiment results to understand:
- Which model won and why
- How accurate the forecasts are (RMSE, MAE, MAPE)
- What features drive predictions (feature importance)
- Where the model struggles (residual analysis)

This notebook generates the artifacts for your portfolio README and employer walkthrough.

In [ ]:
# === Setup ===
import os, sys, json
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')

# Load job info from training step
job_info_path = Path('../data/output/automl_job_info.json')
if job_info_path.exists():
    with open(job_info_path) as f:
        job_info = json.load(f)
    JOB_NAME = job_info['job_name']
    print(f'Job name: {JOB_NAME}')
else:
    JOB_NAME = input('Enter AutoML job name: ')

In [ ]:
# Connect to Azure ML
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=os.getenv('AZURE_SUBSCRIPTION_ID'),
    resource_group_name=os.getenv('AZURE_RESOURCE_GROUP'),
    workspace_name=os.getenv('AML_WORKSPACE_NAME'),
)
print('✓ Connected to Azure ML')

## 1. Model Leaderboard

AutoML tried multiple algorithms in parallel. Let's see how they ranked.

**Common winners for retail forecasting:**
- **LightGBM**: Handles mixed features well (numerical + categorical)
- **Prophet**: Good with strong seasonality + holidays
- **TCNForecaster**: Deep learning — can capture complex temporal patterns

In [ ]:
# Extract the leaderboard from child runs
from src.model_evaluate import generate_leaderboard

leaderboard = generate_leaderboard(ml_client, JOB_NAME)

# Display the full leaderboard
display(leaderboard[['rank', 'algorithm', 'score', 'duration_seconds']].head(15))

In [ ]:
# Visualize the leaderboard
fig, ax = plt.subplots(figsize=(12, 6))

top_models = leaderboard.head(10)
colors = ['#059669' if i == 0 else '#2563EB' for i in range(len(top_models))]

bars = ax.barh(
    top_models['algorithm'][::-1],  # Reverse for top at top
    top_models['score'][::-1],
    color=colors[::-1],
    alpha=0.8
)

ax.set_title('AutoML Model Leaderboard (Top 10)', fontsize=16, fontweight='bold')
ax.set_xlabel('Normalized RMSE (lower = better)')

# Highlight the winner
ax.annotate('★ Winner', xy=(top_models.iloc[0]['score'], len(top_models)-1),
           fontsize=12, color='#059669', fontweight='bold',
           xytext=(10, 0), textcoords='offset points')

plt.tight_layout()
plt.savefig('../screenshots/automl_leaderboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved to screenshots/automl_leaderboard.png')

## 2. Feature Importance

Feature importance answers the question: **"What drives our sales predictions?"**

This is one of the most valuable charts for an employer — it shows you
can interpret models, not just build them.

In [ ]:
from src.model_evaluate import plot_feature_importance

# Generate the feature importance visualization
plot_feature_importance(ml_client, JOB_NAME)
print('\n✓ Feature importance chart saved to screenshots/')

## 3. Generate Demo Visualizations

Run the full evaluation suite to generate all charts for the portfolio.

In [ ]:
# Generate all demo charts (works without Azure connection)
!cd .. && python src/model_evaluate.py --demo

print('\n✓ All evaluation charts generated in screenshots/')
print('  - forecast_vs_actuals.png')
print('  - feature_importance.png')
print('  - residual_analysis.png')

## 4. Model Selection Rationale

### Why the winning model works for retail forecasting:

**If LightGBM won (most likely):**
- Handles mixed feature types natively (categorical stores + numerical prices)
- Captures non-linear relationships (e.g., promotions have diminishing returns)
- Fast training + low memory = scales to millions of rows
- Built-in handling of missing values

**If Prophet won:**
- Explicitly models trend + seasonality + holidays
- Robust to missing data and outliers
- Interpretable decomposition (you can show trend/seasonal components)

**If TCNForecaster won:**
- Temporal Convolutional Network captures complex sequential patterns
- Good at multi-step forecasting (90 days)
- Can learn long-range dependencies automatically

### Key Metrics to Report:
| Metric | What It Means | Target |
|--------|--------------|--------|
| RMSE | Average prediction error (penalizes large misses) | < 15% of avg sales |
| MAPE | Percentage error (scale-independent) | < 20% |
| R² | Variance explained by model | > 0.75 |

**Next step:** `05_endpoint_scoring.ipynb` — deploy and generate forecasts.